# Bronze — Crypto Landing to Delta Table

Reads daily OHLCV CSVs from the landing zone and writes them into a single Delta table
at `s3a://warehouse/bronze/crypto`.

**Each run truncates the table** and reloads it from whatever CSVs are currently in the
landing zone — the Bronze table always reflects the current landing zone state.

**Schema added vs. landing zone:**
- `base_asset` — asset being traded (e.g. `BTC` from `BTCUSDT`)
- `quote_asset` — pricing currency (e.g. `USDT` from `BTCUSDT`)

**Parameters (Cell 2):**
- `SYMBOLS` — list of trading pairs matching CSV filenames in `LANDING_DIR`
- `LANDING_DIR` — path to the landing zone inside the container
- `BRONZE_PATH` — destination Delta table path on MinIO

In [1]:
SYMBOLS     = ["BTCUSDT", "ETHUSDT", "SOLUSDT", "XRPUSDT", "ADAUSDT"]
LANDING_DIR = "/home/landing/crypto/daily"
BRONZE_PATH = "s3a://warehouse/bronze/crypto"

## Setup

In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, col
from pyspark.sql.types import DateType, DoubleType
from delta import configure_spark_with_delta_pip

MINIO_ENDPOINT   = os.environ["MINIO_ENDPOINT"]
MINIO_ACCESS_KEY = os.environ["MINIO_ACCESS_KEY"]
MINIO_SECRET_KEY = os.environ["MINIO_SECRET_KEY"]

builder = (
    SparkSession.builder
    .appName("bronze-crypto")
    .master("spark://spark-master:7077")
    .config("spark.hadoop.fs.s3a.endpoint",               MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key",             MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key",             MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access",      "true")
    .config("spark.hadoop.fs.s3a.impl",                   "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version : {spark.version}")
print(f"MinIO endpoint: {MINIO_ENDPOINT}")
print(f"Bronze path   : {BRONZE_PATH}")

:: loading settings :: url = jar:file:/opt/bitnami/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /opt/bitnami/spark/.ivy2/cache
The jars for the packages stored in: /opt/bitnami/spark/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-abcffd0c-aaa2-4c69-965e-13d3428c5baa;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 127ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     | 

Spark version : 3.5.1
MinIO endpoint: http://minio:9000
Bronze path   : s3a://warehouse/bronze/crypto


## ETL Function

In [3]:
# Recognised quote-asset suffixes, checked longest-first to avoid mis-parsing (e.g. ETHBTC vs BTC)
QUOTE_SUFFIXES = ["USDT", "BUSD", "USDC", "BTC", "ETH", "BNB"]


def ingest_symbol(symbol: str, spark, landing_dir: str, bronze_path: str,
                  write_mode: str = "append") -> int:
    """
    Reads the landing CSV for `symbol`, enriches it with base_asset / quote_asset
    derived columns and casts all types. Writes to the Bronze Delta table using
    `write_mode` ('overwrite' for the first symbol to truncate, 'append' for the rest).
    Returns the number of rows written.
    """
    # --- Parse symbol --------------------------------------------------------
    quote = next((q for q in QUOTE_SUFFIXES if symbol.endswith(q)), None)
    if quote is None:
        raise ValueError(f"Cannot parse quote asset from symbol: {symbol!r}")
    base = symbol[: -len(quote)]

    # --- Read CSV ------------------------------------------------------------
    csv_path = f"{landing_dir}/{symbol}.csv"
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(csv_path)
    )

    # --- Enrich & cast -------------------------------------------------------
    df = (
        df
        .withColumn("base_asset",  lit(base))
        .withColumn("quote_asset", lit(quote))
        .withColumn("date",   col("date").cast(DateType()))
        .withColumn("open",   col("open").cast(DoubleType()))
        .withColumn("high",   col("high").cast(DoubleType()))
        .withColumn("low",    col("low").cast(DoubleType()))
        .withColumn("close",  col("close").cast(DoubleType()))
        .withColumn("volume", col("volume").cast(DoubleType()))
        .select("symbol", "base_asset", "quote_asset",
                "date", "open", "high", "low", "close", "volume")
    )

    # --- Write ---------------------------------------------------------------
    row_count = df.count()
    df.write.format("delta").mode(write_mode).save(bronze_path)
    return row_count


print("Function defined.")

Function defined.


## Run ETL

In [4]:
results = []
errors  = []

print(f"Ingesting {len(SYMBOLS)} symbol(s) into {BRONZE_PATH}")
print(f"(First symbol truncates the table; remaining symbols append.)\n")

for i, symbol in enumerate(SYMBOLS):
    write_mode = "overwrite" if i == 0 else "append"
    print(f"  [{i+1}/{len(SYMBOLS)}] {symbol} ({write_mode}) ...", end=" ", flush=True)
    try:
        n = ingest_symbol(symbol, spark, LANDING_DIR, BRONZE_PATH, write_mode)
        results.append({"symbol": symbol, "rows": n, "status": "ok"})
        print(f"{n} rows written.")
    except Exception as exc:
        print(f"ERROR: {exc}")
        errors.append({"symbol": symbol, "status": "error", "error": str(exc)})

print(f"\nDone. {len(results)} succeeded, {len(errors)} failed.")

Ingesting 5 symbol(s) into s3a://warehouse/bronze/crypto
(First symbol truncates the table; remaining symbols append.)

  [1/5] BTCUSDT (overwrite) ... 

26/03/01 15:14:03 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


365 rows written.
  [2/5] ETHUSDT (append) ... 365 rows written.
  [3/5] SOLUSDT (append) ... 365 rows written.
  [4/5] XRPUSDT (append) ... 365 rows written.
  [5/5] ADAUSDT (append) ... 365 rows written.

Done. 5 succeeded, 0 failed.


## Verify

In [6]:
df_bronze = spark.read.format("delta").load(BRONZE_PATH)

print("Schema:")
df_bronze.printSchema()

total = df_bronze.count()
symbols = [r[0] for r in df_bronze.select("symbol").distinct().orderBy("symbol").collect()]
print(f"Total rows : {total}")
print(f"Symbols    : {symbols}")

print("\nDate range per symbol:")
df_bronze.groupBy("symbol", "base_asset", "quote_asset") \
         .agg({"date": "min", "date": "max"}) \
         .orderBy("symbol") \
         .show(truncate=False)

print("Sample (5 rows):")
df_bronze.orderBy("symbol", "date").show(5, truncate=False)

Schema:
root
 |-- symbol: string (nullable = true)
 |-- base_asset: string (nullable = true)
 |-- quote_asset: string (nullable = true)
 |-- date: date (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: double (nullable = true)

Total rows : 1825
Symbols    : ['ADAUSDT', 'BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'XRPUSDT']

Date range per symbol:
+-------+----------+-----------+----------+
|symbol |base_asset|quote_asset|max(date) |
+-------+----------+-----------+----------+
|ADAUSDT|ADA       |USDT       |2026-03-01|
|BTCUSDT|BTC       |USDT       |2026-03-01|
|ETHUSDT|ETH       |USDT       |2026-03-01|
|SOLUSDT|SOL       |USDT       |2026-03-01|
|XRPUSDT|XRP       |USDT       |2026-03-01|
+-------+----------+-----------+----------+

Sample (5 rows):
+-------+----------+-----------+----------+------+------+------+------+--------------+
|symbol |base_asset|quote_asset|dat